## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **ca15f266-6792-4549-8a84-9ee541427d61**


## Dataset

# intratrait


## Description

This data set was used to test whether species specialized to high elevations or with narrow elevational ranges show more conservative &#40;i.e. less variable&#41; trait responses across their elevational distribution, or in response to neighbours, than species from lower elevations or with wider elevational ranges. We did so by studying intraspecific trait variation of 66 species along 40 elevational gradients in four countries &#40;Switzerland, Australia, New Zealand, China&#41; in both hemispheres. As an indication of potential neighbour interactions that could drive trait variation, we also analysed plant species’ height ratio, its height relative to its nearest neighbour.<br>The following traits and parameters were measured and are available in this data set:<br>As an indication of plant stature, we measured vegetative and generative height, where vegetative height was distance from soil to highest vegetative leaf and generative height was distance to the highest point on the reproductive shoot. As a measure of reproductive investment, we noted the presence of flowers on the randomly chosen individuals &#40;see below&#41;. As a measure of individual and genet basal area, we measured individual plant and patch diameters, in two dimensions &#40;along the largest diameter and perpendicular to it&#41;. In clonal plant species, plant diameter was equivalent to an individual rosette, whereas patch diameter referred to the whole genet and could represent the size of a tuft, tussock or cushion. For genera with more singular growth forms &#40;e.g., some Gentiana species&#41; plant and patch diameter were the same. The two diameter measurements were made at right angles, allowing estimates of patch and plant areas to be calculated as an ellipse &#40;i.e., area = 0.5 a 0.5 b Π&#41;. All traits were measured on ten randomly selected individuals per site. Flower count data was considered in a binary fashion on a per individual basis &#40;because for some species individuals only produce one flower when flowering&#41; so that the presence or absence of flower&#40;s&#41; was a nominal value between 0 and 10 for each species at each site. We then collected at least three leaves &#40;up to 30 for small and light leaves&#41; from each of the first three individuals selected from each species for determination of leaf dry matter content &#40;LDMC&#41; and specific leaf area &#40;SLA&#41;. For calculations of LDMC and SLA, fresh leaves were scanned on a flatbed scanner to determine leaf area. Leaves were then weighed on a balance to a precision of +/- 0.001g, prior to being air dried and reweighed with a balance to a precision of +/- 0.0001g. LDMC was calculated by dividing dry leaf mass by fresh leaf mass. SLA was calculated by dividing leaf area by dry leaf mass. Additionally, within an area of 10 cm diameter around the target individual, we determined the tallest neighbouring species and measured its vegetative and generative height, and estimated the percent cover of the target species, other vegetation, rock, and bare soil. <br>For more details see Rixen et al. 2022, Journal of Ecology.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/intratrait)


## Metadata

- **identifier:** 10.16904&#95;ENVIDAT.269@envidat
- **publisher.name:** EnviDat
- **organization.display&#95;name:** EnviDat: das Umwelt-Datenportal
- **organization.url:** https://www.envidat.ch/
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** climate-change, elevation-range, plant-traits, cold-adapted-plants, neighbour-interactions, elevation-gradient
- **issued:** 2022-01-01 00:00:00
- **metadata&#95;created:** 2026-04-23T14:34:09.820147
- **metadata&#95;modified:** 2026-11-14T02:47:27.942686
- **Resource id:** 666b85b8-f216-4427-85e0-ac1c1d251096
- **Resource name:** 
- **Resource package&#95;id:** ca15f266-6792-4549-8a84-9ee541427d61
- **Resource description:** Resource Title: Rixen&#95;etal&#95;2022&#95;JEcol&#95;AllTraitData<br>    Size: 3679833 bytes<br>    Format: text/csv<br>    URL: https://www.envidat.ch/dataset/58a5f3e7-89e3-41c9-a880-5971ecaceab5/resource/d6845438-a41c-49a6-be0a-254e258a9acd/download/rixen&#95;etal&#95;2022&#95;jecol&#95;alltraitdata.csv<br>    Description: Plant trait data along elevation gradients on mountains
- **Resource issued:** 2022-01-01 00:00:00
- **Resource modified:** 
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** http://www.opendefinition.org/licenses/cc-zero


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **666b85b8-f216-4427-85e0-ac1c1d251096** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://www.envidat.ch/dataset/58a5f3e7-89e3-41c9-a880-5971ecaceab5/resource/d6845438-a41c-49a6-be0a-254e258a9acd/download/rixen_etal_2022_jecol_alltraitdata.csv")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** envidat@wsl.ch &#124; EnviDat
